In [1]:
import os
import sqlite3
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(root / ".env", override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")

db_path = root / "data" / "novamind.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
SELECT MAX(campaign_id) FROM performance_metrics
""")
latest_campaign_id = cursor.fetchone()[0]

cursor.execute("""
SELECT persona, open_rate, click_rate, unsubscribe_rate, subject_line_style, content_angle, cta_type, weighted_score
FROM performance_metrics
WHERE campaign_id = ?
ORDER BY weighted_score DESC
""", (latest_campaign_id,))

rows = cursor.fetchall()

print("Using campaign_id:", latest_campaign_id)
for row in rows:
    print(row)

Using campaign_id: 15
('Operations Manager', 0.46, 0.16, 0.005, 'process-focused', 'workflow efficiency', 'see workflow example', 0.2335)
('Account / Client Services Lead', 0.43, 0.13, 0.007, 'client-service-focused', 'client visibility and responsiveness', 'see delivery example', 0.2063)
('Agency Founder', 0.41, 0.12, 0.01, 'growth-focused', 'scaling without headcount', 'book demo', 0.194)
('Strategy / Marketing Lead', 0.4, 0.11, 0.009, 'planning-focused', 'strategic leverage and execution speed', 'read use case', 0.1851)
('Creative Lead', 0.38, 0.09, 0.015, 'creative-time-focused', 'protecting creative time', 'read blog', 0.1665)


In [2]:
metrics_text = "\n".join([
    f"Persona: {persona}, Open Rate: {open_rate:.3f}, Click Rate: {click_rate:.3f}, "
    f"Unsubscribe Rate: {unsubscribe_rate:.3f}, Subject Line Style: {subject_line_style}, "
    f"Content Angle: {content_angle}, CTA Type: {cta_type}, Weighted Score: {weighted_score:.4f}"
    for persona, open_rate, click_rate, unsubscribe_rate, subject_line_style, content_angle, cta_type, weighted_score in rows
])

prompt = f"""
You are a growth analyst for NovaMind.

Below is newsletter performance by persona for one campaign.

{metrics_text}

Write a short performance summary in 5 to 7 sentences.

Rules:
- Be specific
- Identify the strongest persona
- Explain what likely worked
- Explain what underperformed
- Recommend one improvement for the next campaign
- Keep it concise and practical
"""

response = client.responses.create(
    model=MODEL,
    input=prompt
)

summary_text = response.output_text
print(summary_text)

Operations Manager was the strongest-performing persona, leading the campaign with a 46.0% open rate, 16.0% click rate, just a 0.5% unsubscribe rate, and the highest weighted score at 0.2335. The process-focused subject line and workflow efficiency angle likely worked because they tied NovaMind’s value directly to a clear operational pain point, and the “see workflow example” CTA gave readers a low-friction next step. Account / Client Services Lead also performed well, with a 43.0% open rate and 13.0% click rate, suggesting that messaging around client visibility and responsiveness is another strong fit. On the lower end, Creative Lead underperformed with the weakest open rate (38.0%), lowest click rate (9.0%), and highest unsubscribe rate (1.5%), indicating that the creative-time-focused framing and blog CTA were less compelling for conversion. Agency Founder and Strategy / Marketing Lead were mid-tier, but both likely lost momentum because their angles were broader and their CTAs (“b

In [3]:
prompt = f"""
You are a growth analyst for NovaMind.

Here is campaign performance data by persona:

{metrics_text}

Task:
Return a valid JSON object with these keys:
- next_blog_topics: array of 3 topic ideas
- best_persona_to_prioritize: string
- subject_line_tests: object with keys "Agency Founder", "Operations Manager", "Creative Lead", "Account / Client Services Lead", "Strategy / Marketing Lead"
- newsletter_improvements: object with keys "Agency Founder", "Operations Manager", "Creative Lead", "Account / Client Services Lead", "Strategy / Marketing Lead"

Rules:
- Base recommendations on the performance data
- Prioritize what is likely to improve click rate without increasing unsubscribe rate
- Be practical, not generic
- Keep each suggestion concise
- Return valid JSON only
"""

response = client.responses.create(
    model=MODEL,
    input=prompt
)

optimization_text = response.output_text
print(optimization_text)

{
  "next_blog_topics": [
    "How operations managers can cut project handoff delays with repeatable AI-assisted workflows",
    "Ways client services teams can improve client visibility and response speed without adding status meetings",
    "How agencies scale delivery capacity without increasing headcount: practical workflow examples"
  ],
  "best_persona_to_prioritize": "Operations Manager",
  "subject_line_tests": {
    "Agency Founder": [
      "Scale delivery without adding headcount",
      "A smarter way to increase agency capacity",
      "How agencies grow without hiring first"
    ],
    "Operations Manager": [
      "Reduce handoff delays across your workflow",
      "A faster way to keep work moving",
      "See the workflow top ops teams use to cut bottlenecks"
    ],
    "Creative Lead": [
      "Protect more creative time each week",
      "Less admin, more time for creative work",
      "How creative teams reduce interruption-heavy work"
    ],
    "Account / Client 

In [4]:
import json

optimization_data = json.loads(optimization_text)

print("Best persona to prioritize:", optimization_data["best_persona_to_prioritize"])
print("\nNext blog topics:")
for topic in optimization_data["next_blog_topics"]:
    print("-", topic)

print("\nSubject line tests:")
for persona, suggestion in optimization_data["subject_line_tests"].items():
    print(f"{persona}: {suggestion}")

Best persona to prioritize: Operations Manager

Next blog topics:
- How operations managers can cut project handoff delays with repeatable AI-assisted workflows
- Ways client services teams can improve client visibility and response speed without adding status meetings
- How agencies scale delivery capacity without increasing headcount: practical workflow examples

Subject line tests:
Agency Founder: ['Scale delivery without adding headcount', 'A smarter way to increase agency capacity', 'How agencies grow without hiring first']
Operations Manager: ['Reduce handoff delays across your workflow', 'A faster way to keep work moving', 'See the workflow top ops teams use to cut bottlenecks']
Creative Lead: ['Protect more creative time each week', 'Less admin, more time for creative work', 'How creative teams reduce interruption-heavy work']
Account / Client Services Lead: ['Give clients better visibility without extra status work', 'Respond faster without adding coordination overhead', 'See 